In [9]:
import pandas as pd
from pathlib import Path   
import numpy as np

In [10]:
result_path = Path("../results/downstream_task")
bias_types = ["less_positive_class"]
metrics = ["AUROC"]
less_bias_strengths = ["0.1"]
method_name_replacer = {
                        "uniform": "Uniform", 
                        "psa": "PSA", 
                        "kmm": "KMM", 
                        "mrs-forest": "MRS", 
                        "fw-mrs-temperature": "FW-MRS",
                        "fw-mrs-temperature-svm": "FW-MRS$_{SVM}$",
                        }
data_set_replacer = {
                    "diabetes": "Diabetes", 
                    "folktables_employment": "Employment", 
                    "folktables_income": "Income",
                    "bank_marketing": "Bank Marketing",
                    "hr_analytics": "HR Analytic", 
                    "german_credit": "German Credit", 
                    "breast_cancer": "Breast Cancer", 
                    "loan_prediction": "Loan",
                    }

In [11]:
aurocs = []
auprcs = []
dict_list = []
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
        for method in method_name_replacer.keys():
            for bias_strength in less_bias_strengths:
                json_file = result_path / dataset / bias_type /  bias_strength/ method / "classification_results.json"
                try:
                    result_file = pd.read_json(str(json_file))
                except FileNotFoundError:
                    continue
                dict_list.append(
                    {
                        "Method": method, "Data Set": dataset, 
                        "AUROC Mean": result_file["random forest auroc"]["mean"], 
                        "AUROC Std": result_file["random forest auroc"]["sd"], 
                        "Bias Type": bias_type, "Bias Strength": bias_strength,
                        "Dropped Samples Mean": result_file["dropped_samples"]["mean"],
                        "Dropped Samples Std": result_file["dropped_samples"]["std"],
                        "RF Domain AUROC Mean": result_file["rf domain auroc"]["mean"],
                        "RF Domain AUROC Std": result_file["rf domain auroc"]["sd"],
                    }
                                )
result_df = pd.DataFrame(data=dict_list)

In [12]:
result_df = result_df.replace(method_name_replacer)
result_df

,Method,Data Set,AUROC Mean,AUROC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std,RF Domain AUROC Mean,RF Domain AUROC Std
0,Uniform,diabetes,0.791498,0.022072,less_positive_class,0.1,0.000000,0.000000,0.524808,0.009243
1,PSA,diabetes,0.787277,0.023148,less_positive_class,0.1,0.000000,0.000000,0.512270,0.003703
2,KMM,diabetes,0.781169,0.021943,less_positive_class,0.1,0.000000,0.000000,0.521981,0.007816
3,MRS,diabetes,0.789326,0.023831,less_positive_class,0.1,39.600000,32.493692,0.522293,0.009748
4,FW-MRS,diabetes,0.786366,0.022672,less_positive_class,0.1,42.700000,34.629612,0.520027,0.009081
5,FW-MRS$_{SVM}$,diabetes,0.781606,0.029615,less_positive_class,0.1,61.400000,33.181923,0.518128,0.007736
6,Uniform,folktables_employment,0.870601,0.010491,less_positive_class,0.1,0.000000,0.000000,0.630367,0.010368
7,PSA,folktables_employment,0.867401,0.010937,less_positive_class,0.1,0.040000,0.280000,0.528106,0.009637
8,KMM,folktables_employment,0.856631,0.013580,less_positive_class,0.1,0.000000,0.000000,0.525253,0.008685
9,MRS,folktables_employment,0.869954,0.009992,less_positive_class,0.1,203.900000,35.061232,0.602020,0.012732


In [13]:
for bias_type in bias_types:
    for bias_strength in less_bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for dataset in data_set_replacer.keys():
            mean_auroc_values = []
            std_auroc_values = []
            for method in result_df["Method"].unique():
                try:
                    mean_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUROC Mean"].iloc[0]
                    mean_auroc_values.append(np.round(mean_auroc, 3))

                    std_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUROC Std"].iloc[0]
                    std_auroc_values.append(np.round(std_auroc, 2))
                except IndexError:
                    mean_auroc_values.append(0)
                    std_auroc_values.append(0)

            print(f"{data_set_replacer[dataset]} \
& ${mean_auroc_values[0]}\\pm{std_auroc_values[0]}$ \
& ${mean_auroc_values[1]}\\pm{std_auroc_values[1]}$ \
& ${mean_auroc_values[2]}\\pm{std_auroc_values[2]}$ \
& ${mean_auroc_values[3]}\\pm{std_auroc_values[3]}$ \
& ${mean_auroc_values[4]}\\pm{std_auroc_values[4]}$ \
& ${mean_auroc_values[5]}\\pm{std_auroc_values[5]}$ \
\\\\")
        print("\n")

less_positive_class, 0.1
Diabetes & $0.791\pm0.02$ & $0.787\pm0.02$ & $0.781\pm0.02$ & $0.789\pm0.02$ & $0.786\pm0.02$ & $0.782\pm0.03$ \\
Employment & $0.871\pm0.01$ & $0.867\pm0.01$ & $0.857\pm0.01$ & $0.87\pm0.01$ & $0.867\pm0.01$ & $0.864\pm0.01$ \\
Income & $0.838\pm0.01$ & $0.831\pm0.01$ & $0.82\pm0.01$ & $0.838\pm0.01$ & $0.834\pm0.01$ & $0.832\pm0.01$ \\
Bank Marketing & $0.847\pm0.02$ & $0.839\pm0.03$ & $0.832\pm0.03$ & $0.846\pm0.02$ & $0.845\pm0.02$ & $0.84\pm0.02$ \\
HR Analytic & $0.753\pm0.02$ & $0.75\pm0.02$ & $0.749\pm0.02$ & $0.751\pm0.02$ & $0.751\pm0.02$ & $0.751\pm0.02$ \\
German Credit & $0.667\pm0.05$ & $0.659\pm0.06$ & $0.649\pm0.05$ & $0.668\pm0.05$ & $0.644\pm0.06$ & $0.632\pm0.07$ \\
Breast Cancer & $0.988\pm0.01$ & $0.988\pm0.01$ & $0.989\pm0.01$ & $0.989\pm0.01$ & $0.98\pm0.01$ & $0.977\pm0.01$ \\
Loan & $0.658\pm0.08$ & $0.628\pm0.1$ & $0.61\pm0.1$ & $0.638\pm0.1$ & $0.612\pm0.09$ & $0.575\pm0.09$ \\




In [14]:
for bias_type in bias_types:
    for bias_strength in less_bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for dataset in data_set_replacer.keys():
            mean_domain_values = []
            std_domain_values = []
            for method in result_df["Method"].unique():
                try:
                    mean_domain = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["RF Domain AUROC Mean"].iloc[0]
                    mean_domain_values.append(np.round(mean_domain, 3))

                    std_domain = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["RF Domain AUROC Std"].iloc[0]
                    std_domain_values.append(np.round(std_domain, 2))
                except IndexError:
                    mean_domain_values.append(0)
                    std_domain_values.append(0)

            print(f"{data_set_replacer[dataset]} \
& ${mean_domain_values[0]}\\pm{std_domain_values[0]}$ \
& ${mean_domain_values[1]}\\pm{std_domain_values[1]}$ \
& ${mean_domain_values[2]}\\pm{std_domain_values[2]}$ \
& ${mean_domain_values[3]}\\pm{std_domain_values[3]}$ \
& ${mean_domain_values[4]}\\pm{std_domain_values[4]}$ \
& ${mean_domain_values[5]}\\pm{std_domain_values[5]}$ \
& \\\\")
        print("\n")

less_positive_class, 0.1
Diabetes & $0.525\pm0.01$ & $0.512\pm0.0$ & $0.522\pm0.01$ & $0.522\pm0.01$ & $0.52\pm0.01$ & $0.518\pm0.01$ & \\
Employment & $0.63\pm0.01$ & $0.528\pm0.01$ & $0.525\pm0.01$ & $0.602\pm0.01$ & $0.604\pm0.01$ & $0.577\pm0.01$ & \\
Income & $0.613\pm0.01$ & $0.519\pm0.01$ & $0.528\pm0.01$ & $0.587\pm0.02$ & $0.586\pm0.02$ & $0.571\pm0.01$ & \\
Bank Marketing & $0.523\pm0.01$ & $0.512\pm0.0$ & $0.521\pm0.01$ & $0.517\pm0.01$ & $0.518\pm0.01$ & $0.518\pm0.01$ & \\
HR Analytic & $0.533\pm0.01$ & $0.511\pm0.0$ & $0.514\pm0.01$ & $0.527\pm0.01$ & $0.525\pm0.01$ & $0.527\pm0.01$ & \\
German Credit & $0.538\pm0.02$ & $0.533\pm0.01$ & $0.547\pm0.02$ & $0.534\pm0.01$ & $0.531\pm0.01$ & $0.533\pm0.01$ & \\
Breast Cancer & $0.731\pm0.02$ & $0.583\pm0.03$ & $0.592\pm0.03$ & $0.649\pm0.03$ & $0.658\pm0.03$ & $0.688\pm0.04$ & \\
Loan & $0.57\pm0.03$ & $0.554\pm0.02$ & $0.585\pm0.02$ & $0.551\pm0.02$ & $0.553\pm0.02$ & $0.552\pm0.02$ & \\




In [15]:
result_df.round(3)

,Method,Data Set,AUROC Mean,AUROC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std,RF Domain AUROC Mean,RF Domain AUROC Std
0,Uniform,diabetes,0.791,0.022,less_positive_class,0.1,0.000,0.000,0.525,0.009
1,PSA,diabetes,0.787,0.023,less_positive_class,0.1,0.000,0.000,0.512,0.004
2,KMM,diabetes,0.781,0.022,less_positive_class,0.1,0.000,0.000,0.522,0.008
3,MRS,diabetes,0.789,0.024,less_positive_class,0.1,39.600,32.494,0.522,0.010
4,FW-MRS,diabetes,0.786,0.023,less_positive_class,0.1,42.700,34.630,0.520,0.009
5,FW-MRS$_{SVM}$,diabetes,0.782,0.030,less_positive_class,0.1,61.400,33.182,0.518,0.008
6,Uniform,folktables_employment,0.871,0.010,less_positive_class,0.1,0.000,0.000,0.630,0.010
7,PSA,folktables_employment,0.867,0.011,less_positive_class,0.1,0.040,0.280,0.528,0.010
8,KMM,folktables_employment,0.857,0.014,less_positive_class,0.1,0.000,0.000,0.525,0.009
9,MRS,folktables_employment,0.870,0.010,less_positive_class,0.1,203.900,35.061,0.602,0.013


In [16]:
result_df["Rank AUROC"] = result_df.round(3).groupby("Data Set")["AUROC Mean"].rank(ascending=False)
result_df["Rank Domain"] = result_df.round(3).groupby("Data Set")["RF Domain AUROC Mean"].rank(ascending=True)
result_df[["Method", "Rank AUROC", "Rank Domain"]].groupby("Method").mean()

,Rank AUROC,Rank Domain
Method,,
FW-MRS,3.8125,3.3125
FW-MRS$_{SVM}$,4.8750,3.1875
KMM,5.0625,3.5625
MRS,1.8750,3.5000
PSA,3.8750,1.6875
Uniform,1.5000,5.7500
